# XD2-Net on Google Colab — experiment runner

Runs `scripts/run_experiment.py` (both models × datasets × buckets) on a Colab GPU.

1. **Runtime → Change runtime type → GPU** (T4 is enough; L4/A100 use bf16 automatically).
2. The code is cloned from GitHub (`BRANCH` below), so **push your branch first**.
3. Outputs go to `MyDrive/xd2net_outputs/<model>/<dataset>/` and survive the runtime.
4. **After a disconnect, just run all cells again.** `--skip-existing` resumes and skips
   buckets that are already finished.

For the hierarchical attention views, open `05_hier_explain.ipynb` afterwards (same setup cell,
`WORKSPACE` defaults to the Drive folder).

In [ ]:
import os, sys, glob, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab: clone the repo into /content (code) and write outputs to Drive (they survive the runtime).
REPO_URL = 'https://github.com/bemali/xd2-net-pytorch.git'
BRANCH = 'feattransformer-self-attention-exp'       # switch to 'main' once merged
OUTPUT_ROOT = None                                   # None = configs' workspace_dir (./outputs)

if IN_COLAB:
    REPO_ROOT = '/content/xd2-net-pytorch'
    if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
        !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
    %pip install -e {REPO_ROOT} --quiet
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/xd2net_outputs'
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
else:
    REPO_ROOT = os.path.abspath('..')

sys.path.insert(0, REPO_ROOT)   # importable without a kernel restart after pip install -e
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT, '| commit:',
      subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip())
print('OUTPUT_ROOT:', OUTPUT_ROOT)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU runtime)')
except ImportError:
    pass

## GPU check

In [ ]:
import torch
if torch.cuda.is_available():
    !nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
    print('bf16 supported:', torch.cuda.is_bf16_supported(), '→ AMP dtype',
          'bfloat16' if torch.cuda.is_bf16_supported() else 'float16 (+ GradScaler)')
else:
    print('WARNING: no GPU. Runtime → Change runtime type → GPU, then rerun.')

## Parameters

`BUCKETS = []` runs every bucket the dataset config lists. `EPOCHS` / `BATCH_SIZE` = `None`
keep the values from `configs/models/<model>.yaml`. A larger batch size makes better use of the
GPU but changes the optimisation; retune `learning_rate` if you change it a lot.

In [ ]:
MODELS     = ['baseline', 'hierarchical']
DATASETS   = ['configs/datasets/bpic2012.yaml']
BUCKETS    = []          # e.g. ['1_5', '1_10']; [] = all
EPOCHS     = None        # None = from config (100, with early stopping)
BATCH_SIZE = None        # None = from config (256)
AMP        = True        # mixed precision on GPU
SEED       = None        # None = from config
WORKSPACE  = OUTPUT_ROOT or './outputs'

## Run

In [ ]:
import shlex

args = ['--model', *MODELS, '--datasets', *DATASETS, '--workspace', WORKSPACE, '--skip-existing']
if BUCKETS:
    args += ['--buckets', *BUCKETS]
if EPOCHS is not None:
    args += ['--epochs', str(EPOCHS)]
if BATCH_SIZE is not None:
    args += ['--batch-size', str(BATCH_SIZE)]
if SEED is not None:
    args += ['--seed', str(SEED)]
args += ['--amp' if AMP else '--no-amp']

cmd = ' '.join(shlex.quote(a) for a in [sys.executable, 'scripts/run_experiment.py', *args])
print(cmd)
!{cmd}

## Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results = pd.read_csv(os.path.join(WORKSPACE, 'results_all.csv'))
results['n'] = results['bucket'].str.split('_').str[1].astype(int)
for name, g in results.groupby('dataset'):
    table = g.pivot(index='n', columns='model', values='roc_auc').sort_index()
    display(table.round(3))
    table.plot(marker='o', figsize=(7, 3), title=f'{name}: ROC AUC per prefix length', grid=True)
    plt.xlabel('prefix length'); plt.tight_layout(); plt.show()